# Task 2: Contracts, nested data and a star schema
CSEN1095 Data Engineering · W26

**Pipeline:** read `bookings.jsonl` → validate against a Pydantic contract → quarantine bad/duplicate bookings → flatten into two tables → build a star schema in DuckDB → save as Parquet.

Nothing in this notebook is typed in by hand: every count is computed from `DATA`.

In [1]:
DATA = "bookings.jsonl"

## Setup
`pydantic` = the contract, `duckdb` = the star schema and SQL, `pandas` = flat tables, `pathlib`/`shutil` = folders.
We import `datetime as dt` (not `from datetime import date`) because our `Event` model has a field *named* `date`, and we don't want it to clash with the type `date`.

In [2]:
import json, math, shutil
import datetime as dt
from collections import Counter
from pathlib import Path
from typing import Literal

import duckdb
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)

# Part A: The contract

### A1. Pydantic models
A Pydantic model is a class whose fields have types and rules. Giving it a dict either returns a clean object or raises `ValidationError` listing everything wrong.

* `Literal[...]` = "must be exactly one of these values" (ticket type, faculty, club, payment, status).
* `Field(ge=1)` = "greater than or equal to 1" (`qty`); `Field(ge=0)` = not negative (`price`).
* `pattern=` = regular expression for the student id. I use `[0-9]` rather than `\d` because `\d` also matches non-Western digits.
* `min_length=1` on the list = at least one ticket.
* A field with no default is **required**, so a missing or `null` event fails automatically, and "event present with all five fields" needs no extra code.
* `model_validator(mode="after")` runs **after** all fields are individually valid, so it can compare several fields. That is how we express the two cross-field rules: *total = Σ qty×price* and *booked_at before event date*.
* `extra="allow"` keeps unknown fields (like the drifting field) instead of dropping them, which we need for Part B3.

In [3]:
class Ticket(BaseModel):
    model_config = ConfigDict(extra="allow")
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str = Field(pattern=r"^[0-9]{2}-[0-9]{4}$")
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: dt.date
    venue: str


class Booking(BaseModel):
    model_config = ConfigDict(extra="allow")
    booking_id: str
    booked_at: dt.datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]

    # cross-field rule 1: total must equal the sum of qty x price
    @model_validator(mode="after")
    def total_matches_tickets(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.005):
            raise ValueError("total does not equal sum of qty x price")
        return self

    # cross-field rule 2: nobody books on/after the event's date
    @model_validator(mode="after")
    def booked_before_event(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked_at is not before the event date")
        return self

### A2. Validate every line → quarantine
Steps: (1) read each non-empty line; (2) a line that isn't valid JSON, or isn't a JSON object, is quarantined right away (we can't even ask it for a `booking_id`); (3) every parsed document goes through `Booking.model_validate`; (4) failures go to quarantine with `booking_id` and a readable **reason**.

A booking can break several rules at once; the **reason** recorded is the *first* error Pydantic reports (fields are checked in the order they are declared). Cross-field rules only run once all the individual fields are fine.

In [4]:
def clean_reason(err):
    """Turn one Pydantic error into a short, countable reason like 'student.id: String should match pattern ...'."""
    loc = ".".join(str(p) for p in err["loc"] if not isinstance(p, int))  # drop list indexes: tickets.0.qty -> tickets.qty
    msg = err["msg"].removeprefix("Value error, ")
    return f"{loc}: {msg}" if loc else msg


# 1. read lines
lines = []
with open(DATA, encoding="utf-8") as f:
    for n, text in enumerate(f, start=1):
        if text.strip():
            lines.append((n, text))

# 2. parse JSON
quarantine, docs = [], []
for n, text in lines:
    try:
        doc = json.loads(text)
    except json.JSONDecodeError:
        quarantine.append({"booking_id": None, "line": n, "reason": "invalid JSON"})
        continue
    if not isinstance(doc, dict):
        quarantine.append({"booking_id": None, "line": n, "reason": "line is not a JSON object"})
        continue
    docs.append((n, doc))

# 3. validate every parsed document against the contract
valid = []   # (line_no, Booking)
for n, doc in docs:
    try:
        valid.append((n, Booking.model_validate(doc)))
    except ValidationError as e:
        quarantine.append({"booking_id": doc.get("booking_id"), "line": n, "reason": clean_reason(e.errors()[0])})

print(f"read {len(lines)} lines | parsed {len(docs)} | passed validation {len(valid)} | quarantined so far {len(quarantine)}")

read 283 lines | parsed 283 | passed validation 262 | quarantined so far 21


### A3. Re-sent bookings (duplicates)
Among the bookings that **passed validation**, keep the first copy of each `booking_id` (file order) and quarantine later copies with the reason `duplicate booking_id`.

**Why after validation?** If we de-duplicated first, an *invalid* first copy would "claim" the id, and a *valid* re-sent copy would be thrown away as a duplicate, so we'd lose a good booking and report the wrong reason. Validating first means only real, usable bookings can claim an id. (The cell after next tests whether swapping the order changes the counts on this file.)

In [5]:
seen, unique = set(), []   # unique = (line_no, Booking) we keep
for n, b in valid:
    if b.booking_id in seen:
        quarantine.append({"booking_id": b.booking_id, "line": n, "reason": "duplicate booking_id"})
    else:
        seen.add(b.booking_id)
        unique.append((n, b))

# save the quarantine
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, default=str) + "\n")

### A4. Report

In [6]:
reasons = Counter(q["reason"] for q in quarantine)

print(f"Lines read:        {len(lines)}")
print(f"Passed (kept):     {len(unique)}")
print(f"Quarantined:       {len(quarantine)}")
print("\nQuarantined per reason:")
report = (pd.Series(reasons, name="bookings").sort_values(ascending=False).rename_axis("reason").reset_index())
display(report)

# sanity: every line is either kept or quarantined, exactly once
assert len(lines) == len(unique) + len(quarantine)

Lines read:        283
Passed (kept):     259
Quarantined:       24

Quarantined per reason:


,reason,bookings
0,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$',4
1,total does not equal sum of qty x price,4
2,tickets.qty: Input should be greater than or equal to 1,3
3,"tickets.type: Input should be 'standard', 'vip' or 'student'",3
4,duplicate booking_id,3
5,booked_at is not before the event date,3
6,event: Input should be a valid dictionary or instance of Event,2
7,"tickets: List should have at least 1 item after validation, not 0",2


**Experiment for Part D: what if duplicates were checked *before* validation?** Same data, swapped order, to see whether the counts change.

In [7]:
# swapped order: dedupe raw documents first (by booking_id), then validate the survivors
seen_raw, dedup_first, dup_first = set(), [], 0
for n, doc in docs:
    bid = doc.get("booking_id")
    if bid in seen_raw:
        dup_first += 1
    else:
        seen_raw.add(bid)
        dedup_first.append((n, doc))

kept_swapped = 0
for n, doc in dedup_first:
    try:
        Booking.model_validate(doc); kept_swapped += 1
    except ValidationError:
        pass

dups_now = reasons.get("duplicate booking_id", 0)
print(f"{'':32}{'validate -> dedupe (mine)':>26}{'dedupe -> validate':>22}")
print(f"{'duplicates quarantined':32}{dups_now:>26}{dup_first:>22}")
print(f"{'bookings left':32}{len(unique):>26}{kept_swapped:>22}")

                                 validate -> dedupe (mine)    dedupe -> validate
duplicates quarantined                                   3                     3
bookings left                                          259                   259


# Part B: Flatten

### B1. Two flat tables
* `bookings`: one row per booking; the nested `student` and `event` objects become `student_*` and `event_*` columns; no `tickets` column. Any extra (drifted) top-level field becomes a column too, and it is empty (`None`) for bookings that don't have it.
* `booking_tickets`: one row per ticket entry, with `amount = qty × price`.

I take each validated `Booking`, turn it into a plain dict with `model_dump()`, and unpack it.

In [8]:
booking_rows, ticket_rows = [], []
for n, b in unique:
    d = b.model_dump()
    row = {}
    for key, value in d.items():
        if key == "tickets":
            continue                                   # tickets go to their own table
        if isinstance(value, dict):                    # student / event -> student_id, event_club, ...
            for sub_key, sub_value in value.items():
                row[f"{key}_{sub_key}"] = sub_value
        else:
            row[key] = value
    booking_rows.append(row)

    for t in d["tickets"]:
        ticket_rows.append({"booking_id": d["booking_id"], "type": t["type"], "qty": t["qty"],
                            "price": t["price"], "amount": t["qty"] * t["price"]})

bookings = pd.DataFrame(booking_rows)
booking_tickets = pd.DataFrame(ticket_rows)

# clean types: UTC timestamps without timezone (so day boundaries are unambiguous), event date as a date
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True).dt.tz_localize(None)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])

print("bookings:", bookings.shape, "| booking_tickets:", booking_tickets.shape)
display(bookings.head(3))
display(booking_tickets.head(3))

bookings: (259, 14) | booking_tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN
1,B0002,2026-09-27 12:29:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN
2,B0003,2026-10-13 20:19:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0


### B2. Assertions

In [9]:
# 1. booking_id is unique in bookings
assert bookings["booking_id"].is_unique, "duplicate booking_id in bookings!"

# 2. for every booking, the ticket amounts add up to the total
amount_per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].to_frame().join(amount_per_booking.rename("sum_amount"))
assert check["sum_amount"].notna().all(), "a booking has no tickets"
assert ((check["total"] - check["sum_amount"]).abs() < 0.005).all(), "amounts do not add up to total"
print("Both assertions passed.")

Both assertions passed.


### B3. Schema drift
**Schema drift** = documents slowly gain (or lose) fields over time. Method: compute the set of all field paths of every parsed document, subtract the paths the contract knows, and for each leftover field find the **earliest `booked_at`** among documents that have it. I look at *all* parsed lines (not only valid ones), because the drift is a property of the data feed, not of whether a booking is valid. I use the earliest `booked_at` (not the first line) because the file isn't sorted by time.

In [10]:
EXPECTED = (set(Booking.model_fields)
            | {f"student.{k}" for k in Student.model_fields}
            | {f"event.{k}" for k in Event.model_fields}
            | {f"tickets[].{k}" for k in Ticket.model_fields})

def field_paths(d, prefix=""):
    out = set()
    for k, v in d.items():
        out.add(prefix + k)
        if isinstance(v, dict):
            out |= field_paths(v, prefix + k + ".")
        elif isinstance(v, list):
            for item in v:
                if isinstance(item, dict):
                    out |= field_paths(item, prefix + k + "[].")
    return out

extra_found = {}   # field path -> list of (booked_at, booking_id, line)
for n, doc in docs:
    for p in field_paths(doc) - EXPECTED:
        extra_found.setdefault(p, []).append((doc.get("booked_at"), doc.get("booking_id"), n))

all_booked_at = pd.to_datetime(pd.Series([d.get("booked_at") for _, d in docs]), utc=True, errors="coerce")

if not extra_found:
    print("No schema drift found.")
for path, hits in extra_found.items():
    times = pd.to_datetime(pd.Series([h[0] for h in hits]), utc=True, errors="coerce")
    first = times.min()
    later_docs = (all_booked_at >= first).sum()
    print(f"Drifted field: '{path}'")
    print(f"  present in {len(hits)} of {len(docs)} documents")
    print(f"  earliest booked_at with the field: {first}")
    print(f"  documents booked on/after that moment: {later_docs}  (of which {later_docs - len(hits)} lack the field)")
    if path in bookings.columns:
        print("  values:", bookings[path].value_counts(dropna=False).to_dict())

Drifted field: 'promo_code'
  present in 193 of 283 documents
  earliest booked_at with the field: 2026-10-20 01:05:00+00:00
  documents booked on/after that moment: 194  (of which 1 lack the field)
  values: {nan: 172, 'CLUB10': 47, 'EARLY': 40}


**What the code does with documents that don't have the field:** `extra="allow"` on the models means the field never causes a rejection, and a booking without it is perfectly valid. When flattening, `pd.DataFrame` builds columns from the union of keys, so such bookings simply get an empty value (`None`/`NaN`) in that column. Nothing is dropped and nothing is invented.

# Part C: Model and store (DuckDB star schema)

### C1. Dimensions and fact
Grain of the fact: **one row per ticket entry**.

* `dim_event`: one row per `event_id`, surrogate key `event_key` (1, 2, 3 ... assigned by `ROW_NUMBER()`).
* `dim_student`: one row per `student_id`, surrogate key `student_key`.
* `fact_ticket_sales`: keys + measures (`qty`, `price`, `amount`) + `status` + `date_key` (`YYYYMMDD` from `booked_at`, in UTC).

If one event or student appeared with two different descriptions, I keep the **first-booked** version, so there's exactly one row per natural key. The cell below checks whether that ever matters.

Note the booking's `total` is deliberately **not** in the fact (see README, Part D).

In [11]:
# does any event / student appear with conflicting descriptions?
ev_conflicts = bookings.groupby("event_id")[["event_title", "event_club", "event_date", "event_venue"]].nunique().gt(1).any(axis=1).sum()
st_conflicts = bookings.groupby("student_id")[["student_name", "student_faculty"]].nunique().gt(1).any(axis=1).sum()
print(f"events with conflicting attributes: {ev_conflicts} | students with conflicting attributes: {st_conflicts}")

events with conflicting attributes: 0 | students with conflicting attributes: 0


In [12]:
con = duckdb.connect()            # in-memory database
con.register("bookings", bookings)
con.register("booking_tickets", booking_tickets)

con.execute("""
CREATE TABLE dim_event AS
SELECT ROW_NUMBER() OVER (ORDER BY event_id) AS event_key,
       event_id, event_title, event_club,
       CAST(event_date AS DATE) AS event_date, event_venue
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY booked_at, booking_id) AS rn FROM bookings)
WHERE rn = 1
""")

con.execute("""
CREATE TABLE dim_student AS
SELECT ROW_NUMBER() OVER (ORDER BY student_id) AS student_key,
       student_id, student_name, student_faculty
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY student_id ORDER BY booked_at, booking_id) AS rn FROM bookings)
WHERE rn = 1
""")

con.execute("""
CREATE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       t.type   AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key
FROM booking_tickets t
JOIN bookings     b ON b.booking_id = t.booking_id
JOIN dim_event    e ON e.event_id   = b.event_id
JOIN dim_student  s ON s.student_id = b.student_id
""")

for name in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(name, con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0], "rows")
display(con.execute("SELECT * FROM dim_event ORDER BY event_key").df())
display(con.execute("SELECT * FROM fact_ticket_sales LIMIT 5").df())

dim_event 8 rows
dim_student 103 rows
fact_ticket_sales 314 rows


,event_key,event_id,event_title,event_club,event_date,event_venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,date_key
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101


### C2. Assertions
If a join silently dropped or multiplied rows, these would catch it.

In [13]:
n_fact = con.execute("SELECT COUNT(*) FROM fact_ticket_sales").fetchone()[0]
assert n_fact == len(booking_tickets), f"fact has {n_fact} rows, booking_tickets has {len(booking_tickets)}"

fact_amount = con.execute("SELECT SUM(amount) FROM fact_ticket_sales").fetchone()[0]
bookings_total = bookings["total"].sum()
assert math.isclose(fact_amount, bookings_total, abs_tol=0.01), (fact_amount, bookings_total)
print(f"Both assertions passed: {n_fact} fact rows; SUM(amount) = {fact_amount:,.2f} = SUM(total) = {bookings_total:,.2f}")

Both assertions passed: 314 fact rows; SUM(amount) = 31,285.00 = SUM(total) = 31,285.00


### C3. Answer from the star: confirmed bookings, per club and ticket type
Join the fact to `dim_event` through the surrogate key, keep `status = 'confirmed'`, then sum.

In [14]:
answer = con.execute("""
SELECT e.event_club        AS club,
       f.ticket_type,
       CAST(SUM(f.qty) AS BIGINT) AS tickets_sold,
       ROUND(SUM(f.amount), 2) AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e ON e.event_key = f.event_key
WHERE f.status = 'confirmed'
GROUP BY e.event_club, f.ticket_type
ORDER BY e.event_club, f.ticket_type
""").df()
display(answer)
print("Total confirmed tickets:", answer["tickets_sold"].sum(), "| total confirmed revenue:", round(answer["revenue_egp"].sum(), 2))

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45,900.0
1,Chess,student,44,440.0
2,Debate,standard,65,1950.0
3,Debate,student,34,510.0
4,Debate,vip,40,3000.0
5,Football,standard,64,1600.0
6,Music,standard,67,2930.0
7,Music,student,20,1000.0
8,Music,vip,78,9990.0
9,Robotics,standard,60,2400.0


Total confirmed tickets: 577 | total confirmed revenue: 28320.0


**Why `total` is not in the fact:** `total` belongs to the *whole booking*, but the fact has one row per *ticket entry*. A booking with 2 entries would repeat its total on 2 rows, and `SUM(total)` would count it twice. Demonstration:

In [15]:
wrong = con.execute("""
SELECT SUM(b.total) FROM fact_ticket_sales f JOIN bookings b ON b.booking_id = f.booking_id
""").fetchone()[0]
multi = booking_tickets.groupby("booking_id").size().gt(1).sum()
print(f"SUM(total) over the fact rows : {wrong:,.2f}   <- inflated")
print(f"True SUM(total) over bookings : {bookings_total:,.2f}")
print(f"Overcount                     : {wrong - bookings_total:,.2f}  (from {multi} bookings with 2+ ticket entries)")

SUM(total) over the fact rows : 42,000.00   <- inflated
True SUM(total) over bookings : 31,285.00
Overcount                     : 10,715.00  (from 55 bookings with 2+ ticket entries)


### C4. Save as Parquet
* `star/dim_event.parquet` and `star/dim_student.parquet`: one file each.
* `star/fact_ticket_sales/booked_month=YYYY-MM/...`: the fact **partitioned** by the month the booking was made. DuckDB's `PARTITION_BY` creates one folder per month. The month is derived from `date_key`.

In [16]:
shutil.rmtree("star", ignore_errors=True)   # so re-running the notebook never leaves stale files
Path("star").mkdir()

con.execute("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
con.execute("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")
con.execute("""
COPY (
  SELECT *,
         substr(CAST(date_key AS VARCHAR), 1, 4) || '-' || substr(CAST(date_key AS VARCHAR), 5, 2) AS booked_month
  FROM fact_ticket_sales
) TO 'star/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

for p in sorted(Path("star").rglob("*.parquet")):
    print(p)

# read the partitioned fact back and confirm nothing was lost
back = con.execute("SELECT COUNT(*), SUM(amount) FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning=true)").fetchone()
assert back[0] == n_fact and math.isclose(back[1], fact_amount, abs_tol=0.01)
print("Read-back OK:", back)

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
Read-back OK: (314, 31285.0)
